# Model 2 - CNN Sequential

Notebook huấn luyện và đánh giá đầy đủ mô hình CNN tuần tự.

In [ ]:
# 1. Import và cấu hình
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / 'data').exists() and PROJECT_DIR != PROJECT_DIR.parent:
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data' / 'VN_trash_classification_preprocessing'
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Data:', DATA_DIR)

In [ ]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(), normalize,
])
test_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])
train_dataset = ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = ImageFolder(DATA_DIR / 'val', transform=test_transform)
test_dataset = ImageFolder(DATA_DIR / 'test', transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
class_names = train_dataset.classes
print('Classes:', class_names)
print('Train:', len(train_dataset), '| Validation:', len(val_dataset), '| Test:', len(test_dataset))

In [ ]:
# 3. Tạo CNN Sequential
class CNNSequential(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(),
            nn.Conv2d(48, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(),
            nn.Conv2d(96, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(96, 192, 3, padding=1), nn.BatchNorm2d(192), nn.ReLU(),
            nn.Conv2d(192, 192, 3, padding=1), nn.BatchNorm2d(192), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(192, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))

model = CNNSequential(len(class_names)).to(device)
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))

In [ ]:
# 4. Loss, optimizer và scheduler
class_counts = Counter(train_dataset.targets)
class_weights = torch.tensor([len(train_dataset) / (len(class_names) * class_counts[i]) for i in range(len(class_names))], dtype=torch.float32, device=device)
loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.01)

In [ ]:
# 5. Train và validation
def run_epoch(data_loader, training=False):
    model.train(training)
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_function(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, total_correct / total_samples

best_val_accuracy, best_state, best_epoch = 0.0, None, 0
for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader)
    scheduler.step()
    if val_accuracy > best_val_accuracy:
        best_val_accuracy, best_epoch = val_accuracy, epoch + 1
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    print(f'Epoch {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}')

model.load_state_dict(best_state)
checkpoint_path = PROJECT_DIR / 'src' / 'model2' / 'checkpoints' / 'cnn_sequential_notebook.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_val_accuracy:.2%}')
print('Checkpoint:', checkpoint_path)

In [ ]:
# 6. Test và đánh giá
model.eval()
true_labels, predicted_labels = [], []
test_loss = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        test_loss += loss_function(outputs, labels).item() * labels.size(0)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(outputs.argmax(dim=1).cpu().tolist())

accuracy = sum(t == p for t, p in zip(true_labels, predicted_labels)) / len(true_labels)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
_, _, weighted_f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted', zero_division=0)
print('\nTest Result')
print('Test Loss:', f'{test_loss / len(test_dataset):.4f}')
print('Test Accuracy:', f'{accuracy:.2%}')
print('Macro Precision:', f'{macro_precision:.4f}')
print('Macro Recall:', f'{macro_recall:.4f}')
print('Macro F1:', f'{macro_f1:.4f}')
print('Weighted F1:', f'{weighted_f1:.4f}')
print('Confusion Matrix:\n', confusion_matrix(true_labels, predicted_labels))